# NIA 과제 인도네시아어 말뭉치 LexisNexis Crawler

In [2]:
#https://github.com/SergeyPirogov/webdriver_manager
%pip install selenium
%pip install webdriver_manager 


  Obtaining dependency information for selenium from https://files.pythonhosted.org/packages/0e/59/aae37fa93e2d4292c3148efcc3066c8ecfe5cfaa72bf8c0b1a5614622cf7/selenium-4.15.2-py3-none-any.whl.metadata
  Obtaining dependency information for trio~=0.17 from https://files.pythonhosted.org/packages/39/46/620fbe56f41fa3ccdda2136d947fb9bacce3d1eb163f057f0262a0ddf5e0/trio-0.23.1-py3-none-any.whl.metadata
  Obtaining dependency information for trio-websocket~=0.9 from https://files.pythonhosted.org/packages/48/be/a9ae5f50cad5b6f85bd2574c2c923730098530096e170c1ce7452394d7aa/trio_websocket-0.11.1-py3-none-any.whl.metadata
  Obtaining dependency information for outcome from https://files.pythonhosted.org/packages/55/8b/5ab7257531a5d830fc8000c476e63c935488d74609b50f9384a643ec0a62/outcome-1.3.0.post0-py2.py3-none-any.whl.metadata
     ---------------------------------------- 0.0/58.3 kB ? eta -:--:--
     ---------------------------------------- 58.3/58.3 kB 3.0 MB/s eta 0:00:00
   --------------

In [3]:
#IMPORT LIBRARIES

from selenium import webdriver
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.webdriver.common import by
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
from tqdm import tqdm

import time
import re
import os
import shutil
import math
import sys

In [4]:
#DEFINE FUNCTIONS

def initiate_driver(browsertype, download_path):
    global driver
    global wait
    global actionChains

    service = Service()
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_argument('--no-sandbox')
    capabilities = {
    'browserName': 'chrome',
    'chromeOptions':  {
        'useAutomationExtension': False,
        'forceDevToolsScreenshot': True,
        'args': ['--start-maximized', '--disable-infobars']
    }
    } 
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_settings.popups" : 0,
        "profile.default_content_setting_values.automatic_downloads" : 1,
        "download.prompt_for_download": 'False',
        "download.default_directory" : download_path,
        "download.directory_upgrade": True,
        "safebrowsing.1enabled": True
        }
    chrome_options.add_experimental_option("prefs", prefs)

    if browsertype == 'chromium':
        driver = webdriver.Chrome(executable_path='/snap/bin/chromium.chromedriver', options=chrome_options)
    elif browsertype == 'chrome':
        driver = webdriver.Chrome(service=service, options=chrome_options)

    wait = WebDriverWait(driver, 300)
    actionChains = ActionChains(driver)
    driver.maximize_window()

#로그인 자동화
def login(id, pw) :
    driver.find_element(By.XPATH, '//*[@id="userid"]').send_keys(id)
    driver.find_element(By.XPATH,'//*[@id="password"]').send_keys(pw)
    driver.implicitly_wait(1)
    driver.find_element(By.XPATH,'//input[@id="signInSbmtBtn"]').send_keys(Keys.ENTER)
    time.sleep(10)
    print('Finish login')

#다음 화면으로 넘어갈 때 까지 대기
def waitforspinner():
    wait.until(EC.invisibility_of_element_located((By.XPATH, '//div[@class="box spinner-interactive"]')))

#홈페이지 열기
def open_url(url):
    time.sleep(20)
    driver.get(url)

#캡션 선택
def click_link(navigate_links):
    for link_caption in navigate_links:
        print('Clicking ' + link_caption)
        linkje = wait.until(EC.element_to_be_clickable((By.XPATH, f'//a[@data-text="{link_caption}"]')))
        linkje.click()

def select_source(sources):
    if media == [] or  media[0] == 'all':
        pass
    else:
        print('Selecting source button')
        wait.until(EC.element_to_be_clickable((By.XPATH, "//input[@name='allsources']")))
        driver.find_element_by_xpath("//input[@name='allsources'][@ng-reflect-value='false']").click()
        time.sleep(1)
        if driver.find_element(By.XPATH, "//input[@name='allsources'][@ng-reflect-value='false']").is_selected() == False:
            driver.find_element(By.XPATH, "//input[@name='allsources'][@ng-reflect-value='false']").click()
        for source in sources:
            print('Selecting ' + source)
            linkje = wait.until(EC.element_to_be_clickable((By.XPATH, f"//input[@aria-label='{source}']")))
            linkje.send_keys(Keys.SPACE)

def enter_query(keywords):
    print('Entering Query: ' + keywords)
    driver.find_element(By.XPATH,"//*[@id='searchTerms']").send_keys(keywords)
    driver.implicitly_wait(1)
    driver.find_element(By.XPATH,"//*[@id='searchTerms']").send_keys(Keys.ENTER)
    driver.implicitly_wait(2)
    print('Waiting for results to load')

#인도네시아어 선택
def filter_language(language):
    lan_btn = '//button[@aria-label="Expand Language"]'
    wait.until(EC.presence_of_element_located((By.XPATH, lan_btn)))
    time.sleep(5)
    driver.execute_script("arguments[0].click();",driver.find_element(By.XPATH, lan_btn))
    time.sleep(5)
    print('Indonesian selected')

    #more_btn
    try:
        driver.find_element(By.XPATH, '//div[@class="lessmore"]/button').click()
    except:
        pass

    wait.until(EC.element_to_be_clickable((By.XPATH, f"//label[@title='{language}']")))
    # 인도네시아 선택 버튼 클릭 5번 반복실행
    for _ in range(5):
        driver.find_element(By.XPATH, f"//label[@title='{language}']").click()
        time.sleep(5)  # 5번의 각 클릭 사이마다 5초 휴식

    close_btn = '//button[@aria-label="Collapse Language"]'
    wait.until(EC.presence_of_element_located((By.XPATH, close_btn)))
    driver.execute_script("arguments[0].click();",driver.find_element(By.XPATH, close_btn))
    driver.implicitly_wait(5)
    waitforspinner()
    
    #뉴스 카테고리 선택
def select_category():
    driver.find_element(By.XPATH, '//*[@id="startincontentbutton"]').click()
    driver.find_element(By.XPATH, '//*[@id="kwkmk"]/header/div/div/span/div/div[2]/ul/li[8]/button/span/span[1]').click()
    time.sleep(5)
    print("News category selected")

def filter_publication(type_list):
    print('publication type : ' , type_list)
    pub_btn = '//button[@aria-label="Expand Publication Type"]'
    driver.execute_script("arguments[0].click();",driver.find_element_by_xpath(pub_btn))
    time.sleep(1)

    for type in type_list:
        linkpub = wait.until(EC.element_to_be_clickable((By.XPATH, f"//label[@title='{type}']")))
        linkpub .send_keys(Keys.SPACE)
    
    close_btn = '//button[@aria-label="Collapse Publication Type"]'
    wait.until(EC.presence_of_element_located((By.XPATH, close_btn)))
    driver.execute_script("arguments[0].click();",driver.find_element_by_xpath(close_btn))
    driver.implicitly_wait(3)
    waitforspinner()

    #날짜 지정
def filter_date(start_date, end_date):
    tl_btn = '//button[@aria-label="Expand Timeline"]'
    wait.until(EC.presence_of_element_located((By.XPATH, tl_btn)))
    driver.execute_script("arguments[0].click();",driver.find_element(By.XPATH, tl_btn))
    time.sleep(10)

    #시작년도
    st_date = driver.find_element(By.XPATH, '//div[@class="date-form-group"]/div[1]/datepicker/div/input')
    for x in range(10): #clear()와Keys.CONTRAL + "a"작동이 안됨. for문을 통해 백스페이스로 지우기
        st_date.send_keys(Keys.BACKSPACE)

    st_date.send_keys(start_date)
    #마지막년도
    ed_date = driver.find_element(By.XPATH, '//div[@class="date-form-group"]/div[2]/datepicker/div/input')

    for x in range(10):
        ed_date.send_keys(Keys.BACKSPACE)
    ed_date.send_keys(end_date)
    ed_date.send_keys(Keys.TAB)

    #날짜설정 확인클릭
    dt_btn = driver.find_element(By.XPATH, '//div[@class="date-form-input"]/button')
    driver.execute_script("arguments[0].click();",dt_btn)
    driver.implicitly_wait(3)

    waitforspinner()
    print("Date filtered")

def sort_results(ranking): #Check amount of search results and sort from new to old
    waitforspinner()
    wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="sortbymenulabel"]')))
    driver.find_element(By.XPATH, '//*[@id="sortbymenulabel"]').click()
    driver.implicitly_wait(1)

    editsort = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[@data-action="editsort"]')))
    editsort.click()

    wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="mainSettings"]/div[2]/div/div[2]/div[1]/settingsgeneral/div/section[2]/select/option[3]')))
    driver.find_element(By.XPATH, '//*[@id="mainSettings"]/div[2]/div/div[2]/div[1]/settingsgeneral/div/section[2]/select/option[3]').click()
    driver.implicitly_wait(3)

    if ranking == 'newest':
        driver.find_element(By.XPATH, '//select[@id="sort-by-dropdown-2"]/option[text()="Date (newest-oldest)"]').click()
    elif ranking == 'oldest':
        driver.find_element(By.XPATH, '//select[@id="sort-by-dropdown-2"]/option[text()="Date (oldest-newest)"]').click()
    
    #Click save
    button_save = '//*[@id="saveChangesButton"]'
    wait.until(EC.element_to_be_clickable((By.XPATH, button_save)))
    driver.find_element(By.XPATH, button_save).click()
    
    waitforspinner()

def count_results():
    global last_page
    
    label_results = driver.find_element(By.XPATH, '//*[@id="content"]/header/h2/span').text
    print('Output ' + label_results)
    
    resultcount = int(re.sub('[^0-9]+', '', label_results))
    last_page = math.ceil(resultcount / 50)

    print(last_page)
    print('Found ' + str(resultcount) + ' articles on ' + str(last_page) + ' pages')

def filter_duplicates():
    print('Enabling duplicate filter')
    dropdown_duplicate = "//button[@class='current trigger collapsed icon la-TriangleDownAfter']"
    wait.until(EC.element_to_be_clickable((By.XPATH, dropdown_duplicate)))
    driver.find_element(By.XPATH, dropdown_duplicate).click()
    driver.implicitly_wait(1)
    driver.find_element(By.XPATH, "//button[@data-value='moderate']").click()

    waitforspinner()

def selectallarticles():
    waitforspinner()
    wait.until(EC.visibility_of_element_located((By.XPATH, '//*[@data-action="selectall"]')))
    driver.find_element(By.XPATH, '//*[@data-action="selectall"]').send_keys(Keys.SPACE)
    time.sleep(1)
    if driver.find_element(By.XPATH, '//*[@data-action="selectall"]').is_selected() == False:
        driver.find_element(By.XPATH, '//*[@data-action="selectall"]').send_keys(Keys.SPACE)
    wait.until(EC.visibility_of_element_located((By.XPATH, '//*[@class="select more"]')))
    print("selected all")

def opendownloaddialog():
    wait.until(EC.visibility_of_element_located((By.XPATH, '//*[@data-action="downloadopt"]')))
    driver.find_element(By.XPATH, '//*[@data-action="downloadopt"]').send_keys(Keys.SPACE)

def setdoctype(doctype):
    wait.until(EC.visibility_of_element_located((By.XPATH, '//*[@data-action="download"]')))
    if doctype == 'docx':    
        driver.find_element(By.XPATH, '//*[@id="Docx"]').click() #select Docx format"
    elif doctype == 'rtf':
        driver.find_element(By.XPATH, '//*[@id="Rtf"]').click() #select Rtf format"
    elif doctype == 'pdf':
        driver.find_element(By.XPATH, '//*[@id="Pdf"]').click() #select Rtf format"

def clickdownload():
    wait.until(EC.visibility_of_element_located((By.XPATH, '//*[@data-action="download"]')))
    driver.find_element(By.CSS_SELECTOR, 'body > aside > footer > div > button.button.primary').send_keys(Keys.SPACE)

def downloadf():
    selectallarticles()
    opendownloaddialog()
    
    global download_settings
    download_settings=True
    if download_settings is False:
    
        # Formatting options for LexisNexisTools
        driver.find_element(By.XPATH, '//*[@id="tab-FormattingOptions"]').click()
        time.sleep(1)
        driver.find_element(By.XPATH, '//*[@id="DisplayFirstLastNameEnabled"]').click()
        driver.find_element(By.XPATH, '//*[@id="EmbeddedReferences"]').click()

        download_settings = True
    clickdownload()

    try:
        #Wait for the job description box to appear
        wait.until(EC.visibility_of_element_located((By.XPATH, '//span[@class="status-message success"]')))
        
        #Wait for job description dialog to disappear
        wait.until(EC.invisibility_of_element_located((By.XPATH, '//span[@class="status-message success"]')))
    except:
        driver.refresh()
        selectallarticles()
        opendownloaddialog()
        clickdownload()

    i = 1
    while not len([fname for fname in os.listdir() if re.match('Files\(\d+\).?[.DOCX]', fname)])!=0:
        i += 1
        time.sleep(0.5)
        if i > 50:
            print('keep going')
            break 

def start_downloading():
    global num
    for num in tqdm(range(0, (last_page))):
        time.sleep(1)
        downloadf()
        try:
            driver.find_element(By.XPATH, "//a[@data-action='nextpage']").click()
            waitforspinner()
        except:
            print('Seems like there are no more pages left')
            break
            
def select_50data():
    driver.find_element(By.XPATH, '//*[@id="sortbymenulabel"]').click()
    driver.implicitly_wait(3)
        
    driver.find_element(By.XPATH, '//*[@id="results-list-delivery-toolbar"]/div/ul[2]/li/div/div/button[6]/span').click()
    driver.implicitly_wait(3)
    
    driver.find_element(By.XPATH, '//*[@id="mainSettings"]/div[2]/div/div[2]/div[1]/settingsgeneral/div/section[2]/select/option[3]').click()
    driver.implicitly_wait(3)
    
    driver.find_element(By.XPATH, '//*[@id="saveChangesButton"]').click()
    driver.implicitly_wait(3)
    
    driver.find_element(By.XPATH, '//*[@id="sortbymenulabel"]').click()
    driver.find_element(By.XPATH, '//*[@id="results-list-delivery-toolbar"]/div/ul[2]/li/div/div/button[5]').click()

### 20230802 최신화 => 정상 작동중
### 20230927 최신화

In [ ]:
#ACTION PARAMETERS
start_page = "https://advance.lexis.com" #Open url
second_page = "https://advance.lexis.com/search/?pdmfid=1000516&crid=fe4d65f2-7ba0-4b8b-aec0-726fca843559&pdsearchterms=pada&pdtypeofsearch=searchboxclick&pdsearchtype=SearchBox&pdstartin=hlct%3A1%3A8&pdpsf=&pdqttype=and&pdquerytemplateid=&ecomp=qd1vkkk&earg=pdsf&prid=d1c38a0e-a8f6-4cf9-8aef-95fa84a79b21" 
ID = 'minwoo@arspraxia.com' #Enter ID & Password
Password='#####'
start_date = "01/01/2013" #Set up date for filtering 
end_date = "05/01/2023"
language='Indonesian' #Set up language to Indonesian
download_path= 'C:\\Users\\USER\\Ars Praxia\\[T] 23-05-001 NIA AI 데이터셋 구축 - General\\06-수집데이터\\기사 데이터\\newsdata\\data_pada' #Set up download path

initiate_driver('chrome', download_path) #Type either chrome or chromium depending on webdriver
open_url(start_page) #홈페이지 열기
login(ID, Password) #자동 로그인
open_url(second_page) #두번째 페이지 창 열기
time.sleep(3)
#select_category() #카테고리 선택
filter_date(start_date, end_date) #날짜 지정
filter_language(language) #언어 선택
time.sleep(10)
count_results() #페이지 문서 전체 선택
start_downloading() #선택한 문서 다운로드

Finish login
Date filtered
Indonesian selected
Output News (10,000+)
200
Found 10000 articles on 200 pages


  0%|          | 0/200 [00:00<?, ?it/s]

selected all
keep going


  0%|          | 1/200 [00:56<3:07:21, 56.49s/it]

selected all
keep going


...

 99%|█████████▉| 198/200 [3:23:35<02:02, 61.48s/it]

selected all
keep going


100%|█████████▉| 199/200 [3:24:37<01:01, 61.58s/it]

selected all
keep going


100%|██████████| 200/200 [3:25:53<00:00, 61.77s/it]


In [6]:
#tqdm이 먼저 끝날 시 사용
count_results()
start_downloading()

Output News (10,000+)
200
Found 10000 articles on 200 pages


  0%|          | 0/200 [00:00<?, ?it/s]

selected all
keep going


  0%|          | 1/200 [01:01<3:24:51, 61.76s/it]

selected all
keep going


...

selected all
selected all
keep going


 69%|██████▉   | 138/200 [2:20:39<1:01:15, 59.29s/it]

selected all


 69%|██████▉   | 138/200 [2:20:49<1:03:16, 61.23s/it]


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=119.0.6045.160)
Stacktrace:
	GetHandleVerifier [0x00007FF729CB82B2+55298]
	(No symbol) [0x00007FF729C25E02]
	(No symbol) [0x00007FF729AE05AB]
	(No symbol) [0x00007FF729AC0038]
	(No symbol) [0x00007FF729B46BC7]
	(No symbol) [0x00007FF729B5A15F]
	(No symbol) [0x00007FF729B41E83]
	(No symbol) [0x00007FF729B1670A]
	(No symbol) [0x00007FF729B17964]
	GetHandleVerifier [0x00007FF72A030AAB+3694587]
	GetHandleVerifier [0x00007FF72A08728E+4048862]
	GetHandleVerifier [0x00007FF72A07F173+4015811]
	GetHandleVerifier [0x00007FF729D547D6+695590]
	(No symbol) [0x00007FF729C30CE8]
	(No symbol) [0x00007FF729C2CF34]
	(No symbol) [0x00007FF729C2D062]
	(No symbol) [0x00007FF729C1D3A3]
	BaseThreadInitThunk [0x00007FFE8590257D+29]
	RtlUserThreadStart [0x00007FFE8714AA58+40]
